In [2]:
! pip install langchain langchain-huggingface langchain-community chromadb sentence-transformers

^C


## indexing

In [4]:
import shutil
import os

# Delete the old database folder if it exists
if os.path.exists("./recipe_vector_db"):
    shutil.rmtree("./recipe_vector_db")
    print("Old database deleted successfully!")
else:
    print("No old database found.")

No old database found.


In [5]:
import json
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# --- 1. Load the Data ---
documents = []
with open('/content/drive/MyDrive/rag_recipes_dataset.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        recipe = json.loads(line)
        doc = Document(
            page_content=recipe['text_for_embedding'],
            metadata={"title": recipe['title'], "url": recipe['url']}
        )
        documents.append(doc)

print(f"Loaded {len(documents)} recipes into memory.")

# --- 2. Initialize Model ON THE GPU ---
print("Loading embedding model onto GPU...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cuda'}  # <--- THIS IS THE MAGIC LINE
)

# --- 3. Create Vector DB ---
print("Embedding recipes (this will be fast now!) and saving to ChromaDB...")
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory="./recipe_vector_db"
)

print("Indexing Complete!")

Loaded 31275 recipes into memory.
Loading embedding model onto GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding recipes (this will be fast now!) and saving to ChromaDB...
Indexing Complete!


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


# 1. Load the exact same embedding model (on the GPU)
print("Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cuda'}
)

# 2. Connect to your saved Chroma Database
print("Connecting to ChromaDB...")
vector_db = Chroma(
    persist_directory="./recipe_vector_db",
    embedding_function=embedding_model
)

# 3. Ask a natural language question in Arabic!
query = "الدجاج والبطاطس" # "I want a recipe with chicken and potatoes in the oven"

# 4. Perform the Similarity Search (k=3 means return the top 3 best matches)
print(f"\nSearching for: '{query}'\n")
results = vector_db.similarity_search(query, k=3)

# 5. Print the results to see if the AI found the right recipes
for i, doc in enumerate(results):
    print(f"--- النتيجة (Result) {i+1} ---")
    print(f"Title: {doc.metadata['title']}")
    print(f"URL: {doc.metadata['url']}")
    # Print the first 200 characters of the recipe text to verify
    print(f"Preview: {doc.page_content[:200]}...\n")

Connecting to ChromaDB...

Searching for: 'الدجاج والبطاطس'

--- النتيجة (Result) 1 ---
Title: طريقة الدجاج المحمر
URL: https://www.atyabtabkha.com/recipe/%d8%b7%d8%b1%d9%8a%d9%82%d8%a9-%d8%a7%d9%84%d8%af%d8%ac%d8%a7%d8%ac-%d8%a7%d9%84%d9%85%d8%ad%d9%85%d8%b1-1466759
Preview: عنوان الوصفة: طريقة الدجاج المحمر
المكونات:
دجاج1
بطاطسحجم كبير، مقطعة إلى مكعبات كبيرة  2
بصلحجم كبير، مقطّع إلى حلقات  1
بصلصغير الحجم  5
جزرحجم كبير  2
ملحبحسب الرغبة
فلفل أسودبحسب الرغبة
زعتر برّي...

--- النتيجة (Result) 2 ---
Title: شوربة دجاج بالذرة
URL: https://www.atyabtabkha.com/recipe/%d8%b4%d9%88%d8%b1%d8%a8%d8%a9-%d8%af%d8%ac%d8%a7%d8%ac-%d8%a8%d8%a7%d9%84%d8%b0%d8%b1%d8%a9-1477627
Preview: عنوان الوصفة: شوربة دجاج بالذرة
المكونات:
مرق دجاجثمانية أكواب
صدر دجاجمقطّع إلى مكعبات صغيرة  1
ملحثلاثة أرباع ملعقة صغيرة
فلفل أبيضنصف ملعقة صغيرة
ذرةكوب
نشاء ذرةنصف كوب
بيضمخفوق  1
خطوات التحضير:
1...

--- النتيجة (Result) 3 ---
Title: طريقة مظبي الدجاج في البيت
URL: https://www.atyabtabkha.com/recipe/%d8%b7%d8%

## LLM

In [ ]:
!pip install langchain-groq

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Set your API Key
# Replace 'YOUR_GROQ_API_KEY' with your actual key
os.environ["GROQ_API_KEY"] = "set your api key"

# 2. Initialize the Groq LLM
# llama3-70b-8192 is incredibly fast and speaks excellent Arabic
print("Waking up the Groq LLM...")
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.3 # Low temperature keeps it focused on the facts
)

# --- STAGE 1: QUERY TRANSFORMATION ---
# We create a prompt specifically to translate the dialect to MSA
translation_prompt = PromptTemplate.from_template(
    """أنت مساعد ذكي. قم بتحويل سؤال الطبخ التالي المكتوب بالعامية إلى لغة عربية فصحى دقيقة ومناسبة للبحث في قاعدة بيانات.
    اكتب جملة البحث المترجمة فقط بدون أي مقدمات أو شروحات.

    السؤال الأصلي: {question}
    سؤال البحث بالفصحى:"""
)

# This creates a mini-pipeline just for translation
translator_chain = translation_prompt | llm | StrOutputParser()


# --- STAGE 2: RETRIEVAL ---
# The user's actual question
user_query = "عندي بطاطس وفراخ، إزاي أعمل بيهم صينية في الفرن؟"
print(f"\nUser asked: '{user_query}'")

# Translate it!
msa_query = translator_chain.invoke({"question": user_query})
print(f"LLM Translated to MSA for search: '{msa_query}'")

# Load the exact same embedding model
print("Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cpu'}
)

# Connect to your saved Chroma Database
print("Connecting to ChromaDB...")
vector_db = Chroma(
    persist_directory="./recipe_vector_db",
    embedding_function=embedding_model
)

# Search the Vector DB using the translated query
print("Searching database...")
results = vector_db.similarity_search(msa_query, k=3)

# Combine the retrieved recipes into one big block of text
context_text = ""
for i, doc in enumerate(results):
    context_text += f"\n--- وصفة {i+1}: {doc.metadata['title']} ---\n"
    context_text += f"المصدر: {doc.metadata['url']}\n"
    context_text += f"{doc.page_content}\n"


# --- STAGE 3: GENERATION ---
# Now we ask the LLM to read the recipes and answer the user
generation_prompt = PromptTemplate.from_template(
    """أنت شيف مصري ماهر وودود. استخدم الوصفات التالية للإجابة على سؤال المستخدم.

    القواعد الصارمة:
    1. لا تخترع أي معلومات أو مقادير من خارج الوصفات المرفقة.
    2. إذا لم تكن الإجابة موجودة في الوصفات، قل ببساطة أنك لا تملك وصفة لذلك.
    3. أجب باللهجة المصرية بطريقة جذابة.

    يجب أن يكون هيكل إجابتك مقسماً كالتالي:

    مقدمة ترحيبية سريعة.

    **المقادير:**
    (استخرج المكونات والكميات الدقيقة من الوصفة واكتبها في نقاط هنا)

    **طريقة التحضير:**
    (اكتب الخطوات بالترتيب في قائمة مرقمة هنا)

    ---
    المصدر: (اذكر اسم الوصفة التي استخدمتها)

    السؤال: {question}

    الوصفات المتاحة (Context):
    {context}

    إجابة الشيف:"""
)

generator_chain = generation_prompt | llm | StrOutputParser()

print("\nGenerating final answer...\n")
print("="*50)
final_answer = generator_chain.invoke({
    "question": user_query,
    "context": context_text
})

print(final_answer)
print("="*50)

Waking up the Groq LLM...

User asked: 'عندي بطاطس وفراخ، إزاي أعمل بيهم صينية في الفرن؟'
LLM Translated to MSA for search: 'كيفية تحضير صينية بطاطس وفراخ في الفرن؟'
Searching database...

Generating final answer...

مرحباً بيك! أنا شيف مصري وودود، هساعدك في عمل صينية بطاطس وفراخ في الفرن.

**المقادير:**
* فخذ دجاج: كيلوغرام
* بطاطس: 3
* بصل: 1
* طماطم: 1
* فليفلة خضراء: 1
* فلفل أسود: ملعقة صغيرة
* ملح: ملعقة صغيرة
* ثوم: ملعقة كبيرة
* هال: 4
* ورق غار: 3
* زيت نباتي: ربع كوب

**طريقة التحضير:**
1. في وعاء، نضع البصل، الطماطم، الفليفلة، البهارات، الثوم، الهال وورق الغار.
2. ندعك المكونات بواسطة اليد حتى تمتزج.
3. نضيف البطاطس وندعكها مع باقي المكونات.
4. في صينية، نحمي الزيت على نار متوسطة ونضع الفراخ. نقلبها على النار حتى يتبدل لونها.
5. نضيف خليط الخضار والبهارات على الفراخ وفي الصينية.
6. ندخل الصينية الى فرن محمى على حرارة 200 درجة مئوية حتى تنضج المكونات.

---
المصدر: طريقة عمل صينية بطاطس بالفراخ بدون صلصة
